In [1]:
from SteerEnergyStorage.Formulations.ElectrodeFormulations import ElectrodeFormulation
from SteerEnergyStorage.Constructions.Electrodes import Cathode, Anode
from SteerEnergyStorage.Formulations.Stacks import Stack
from SteerEnergyStorage.Constructions.Cells import StackedPrismaticCell
from SteerEnergyStorage.Materials.ElectrodeMaterials import ActiveMaterial, Binder, ConductiveAdditive
from SteerEnergyStorage.Materials.CurrentCollectors import CurrentCollector
from SteerEnergyStorage.Materials.Separators import Separator
from SteerEnergyStorage.Materials.Electrolytes import Electrolyte
from SteerEnergyStorage.Constructions.Containers import PrismaticCase, PrismaticLid, PrismaticShell

import pandas as pd
import plotly.express as px

In [2]:
# construct cathode
cathode_active_material = ActiveMaterial(name="Faradion_Gen2_4.25V", 
                                            formula="Li2MnSiO4", 
                                            specific_cost=11.26, 
                                            density=4, 
                                            irreversible_capacity_scaling=1, 
                                            reversible_capacity_scaling=1,
                                            half_cell_path='../Data/Cathode_Faradion_Gen2_4.25V.csv')

cathode_conductive_additive = ConductiveAdditive(specific_cost=9, density=1.9, name="SuperC65")

cathode_binder = Binder(name="PVDF", specific_cost=15, density=1.7)

cathode_formulation = ElectrodeFormulation(active_materials={cathode_active_material: 89},
                                            binder={cathode_binder: 5},
                                            conductive_additive={cathode_conductive_additive: 6})

cathode_current_collector = CurrentCollector(name="Aluminium", 
                                             formula="Al", 
                                             specific_cost=6.30, 
                                             density=2.7, 
                                             thickness=15, 
                                             length=16.0,
                                             width=10.8,
                                             bare_tab_area=8.22)

cathode = Cathode(formulation=cathode_formulation,
                    mass_loading=10.68,
                    current_collector=cathode_current_collector,
                    swell_factor=1.0,
                    calender_density=2.60)


In [3]:
# construct anode
anode_active_material = ActiveMaterial(name="Faradion_HC",
                                        formula="Na2Ti3O7",
                                        specific_cost=14.27,
                                        density=1.50,
                                        irreversible_capacity_scaling=1,
                                        reversible_capacity_scaling=1,
                                        half_cell_path='../Data/Anode_Faradion_HC.csv')

anode_conductive_additive = ConductiveAdditive(specific_cost=9, 
                                               density=1.9, 
                                               name="SuperC65")

anode_binder = Binder(name="PVDF", 
                      specific_cost=10, 
                      density=1.7)

anode_formulation = ElectrodeFormulation(active_materials={anode_active_material: 88},
                                         binder={anode_binder: 3},
                                         conductive_additive={anode_conductive_additive: 9})

anode_current_collector = CurrentCollector(name="Copper",
                                            formula="Cu",
                                            specific_cost=6.30,
                                            density=2.70,
                                            thickness=15,
                                            length=16.0,
                                            width=10.8,
                                            bare_tab_area=7.55)

anode = Anode(formulation=anode_formulation,
              mass_loading=5.25,
              current_collector=anode_current_collector,
              swell_factor=1.0,
              calender_density=0.85)


In [4]:
# construct seperator
separator = Separator(thickness=16, 
                      areal_cost=0.9, 
                      density=0.4, 
                      slit_width=11.0, 
                      porosity=47, 
                      fold_length=18.6,
                      name="Separator")

In [5]:

# make the case
prismatic_shell = PrismaticShell(cost=0.14,
                                 mass=110.23,
                                 internal_width=11.3,
                                 internal_length=18.9,
                                 internal_height=1.93,
                                 wall_thickness=0.8)

prismatic_lid = PrismaticLid(cost=0.34,
                             mass=20.56,
                             external_width=1.3, 
                             internal_width=0.8)

prismatic_case = PrismaticCase(lid=prismatic_lid, shell=prismatic_shell)

# construct the stack
stack = prismatic_case.get_optimized_stack(anode=anode,
                                           cathode=cathode,
                                           separator=separator)


In [6]:
# make electrolyte
electrolyte = Electrolyte(name="Electrolyte", 
                          formula="LiPF6", 
                          specific_cost=8.94, 
                          density=1.2)

cell = StackedPrismaticCell(prismatic_case=prismatic_case,
                            stack=stack,
                            electrolyte=electrolyte,
                            electrolyte_overfill=10,
                            voltage_upper_cut_off=4.2,
                            voltage_lower_cut_off=1.0,
                            reversible_capacity=40000,
                            irreversible_capacity=1215,
                            grid_n=200)

In [7]:
figure = cell.get_capacity_voltage_plot(width=900, height=600)
figure.show()

In [8]:
print(f"The number of stacks is {cell.stack.n_stacks}")
print(f"The energy the cell can hold is {cell.energy} Wh")
print(f"The specific energy of the cell is {cell.specific_energy} Wh/kg")
print(f"The cost of the cell is {cell.cost} $")
print(f"The normalized cost of the cell is {cell.normalized_cost} $/kWh")
print(f"The energy density of the cell is {cell.energy_density} Wh/L")

The number of stacks is 71
The energy the cell can hold is 98.14 Wh
The specific energy of the cell is 122.29 Wh/kg
The cost of the cell is 9.98 $
The normalized cost of the cell is 101.64 $/kWh
The energy density of the cell is 193.08 Wh/L


In [9]:
cell.cost_breakdown

{Stack: 8.107, Electrolyte: 1.388, Prismatic Case: 0.48}

In [10]:
figure = cell.get_cost_breakdown_plot(width=1200, height=500)
figure.show() 

In [11]:
figure = cell.get_mass_breakdown_plot(width=1200, height=500)
figure.show()